# 02 — Acceleration

In notebook 01, the ball moved across the screen at constant velocity:

```python
x += dx   # dx never changes
```

It worked. But it looked wrong. No weight. No pull. Nothing real moves like that.

Adding gravity is one line:

```python
dy += gravity   # velocity itself grows each frame
```

That line turns a straight line into a curve. This notebook explains exactly why — and where the formula $y = y_0 + v_0 t + \frac{1}{2}gt^2$ comes from.

**Prerequisites:** [01 — Linear Motion](./01-linear-motion.ipynb). Familiarity with a game loop. Basic algebra.

---

## 1. One Line Changes Everything

In notebook 01, `dy` was a fixed constant. Add one line and it is no longer fixed — it accumulates:

```python
dy += gravity   # velocity grows each frame
y  += dy        # position updates as before
```

The horizontal velocity `dx` stays constant — the ball still moves right at a steady rate. But vertically, `dy` pulls the ball downward a little more every frame. Run the animation and watch the path.

In [ ]:
# FuncAnimation requires an interactive backend — add '%matplotlib widget' above if it renders static
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# --- Physics parameters ---
x0      = 8.0    # starting x position
y0      = 26.0   # starting y position (near top of frame)
dx      = 2.0    # horizontal velocity — constant, same as notebook 01
gravity = -0.15  # acceleration — negative means downward
floor_y = 4.0    # y position of the ground

fig, ax = plt.subplots(figsize=(10, 3))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.set_xlim(0, 100)
ax.set_ylim(0, 30)
ax.set_aspect('equal')
ax.axis('off')

ax.axhline(floor_y, color='#45475a', linewidth=1.5)
ax.text(1, floor_y + 0.8, 'floor', color='#6c7086', fontsize=8, fontfamily='monospace')

ball  = plt.Circle((x0, y0), radius=3, color='#89dceb')
ax.add_patch(ball)
trail_dots = [plt.Circle((0, 0), radius=0.8, color='#89dceb', alpha=0) for _ in range(20)]
for dot in trail_dots:
    ax.add_patch(dot)

info = ax.text(2, 28.5, '', color='#cdd6f4', fontsize=9, fontfamily='monospace')

state = {'x': x0, 'y': y0, 'dy': 0.0, 'landed': False, 'history': []}

def update(frame):
    s = state
    if not s['landed']:
        s['dy'] += gravity        # velocity accumulates — the one new line
        s['y']  += s['dy']
        s['x']  += dx
        if s['y'] <= floor_y:
            s['y']      = floor_y
            s['dy']     = 0.0
            s['landed'] = True
        s['history'].append((s['x'], s['y']))

    ball.set_center((s['x'], s['y']))

    for i, dot in enumerate(trail_dots):
        idx = len(s['history']) - len(trail_dots) + i
        if 0 <= idx < len(s['history']):
            dot.set_center(s['history'][idx])
            dot.set_alpha(0.08 + 0.06 * i)
        else:
            dot.set_alpha(0)

    status = '  [landed]' if s['landed'] else ''
    info.set_text(f't={frame:>3}   dy = {s["dy"]:+.2f}   y = {s["y"]:.1f}{status}')
    return [ball, info] + trail_dots

ani = animation.FuncAnimation(fig, update, frames=70, interval=80, blit=True)
plt.tight_layout()
plt.show()

Watch the `dy` value in the label. In notebook 01 it never changed. Here it grows more negative every frame — the ball accelerates downward. The path is not a straight line.

---

## 2. Why Does One Line Change Everything?

In notebook 01, `dy` was fixed. The position equation was linear:

$$y = y_0 + dy \cdot t$$

`dy` never changed, `t` appears to the first power, the graph is a straight line.

Now `dy` is not fixed. It accumulates:

```python
dy += gravity   # dy changes every frame
y  += dy        # y is updated by a different dy each frame
```

`y` is being moved by a different amount every frame. The rate of change of `y` is itself changing. That is the definition of acceleration — and it is exactly what makes `t` appear squared.

| | Notebook 01 | This notebook |
|---|---|---|
| What stays constant? | velocity (`dy`) | acceleration (`gravity`) |
| What accumulates? | nothing | velocity (`dy`) |
| Output shape | straight line | parabola |
| Equation degree | 1 | 2 |
| Formula | $y = y_0 + vt$ | $y = y_0 + v_0 t + \frac{1}{2}gt^2$ |

One accumulating variable is the difference between degree 1 and degree 2.

In [ ]:
# Watch dy accumulate — using gravity = -2 for easy arithmetic
gravity = -2
dy = 0
y  = 20

print(f'{"Frame":<8}{"dy (velocity)":<18}{"y (position)":<15}{"Δy this frame"}')
print('-' * 55)
prev_y = y
for frame in range(1, 9):
    dy += gravity
    prev_y = y
    y  += dy
    print(f'{frame:<8}{dy:<18}{y:<15}{y - prev_y}')

The Δy column is the key: in notebook 01, every Δy would be identical. Here it grows. The position is not advancing at a constant rate — it is advancing at an ever-increasing rate. That is what a curve looks like in numbers.

---

## 3. The t² Reveal

Let's trace through the loop frame by frame and work out the formula it produces.

**Setup:** `y0 = 20`, initial `dy = 0`, `gravity = -2`.

| Frame `t` | `dy` before | after `dy += gravity` | after `y += dy` | `y` |
|-----------|------------|----------------------|-----------------|-----|
| 0         | 0          | **−2**               | 20 + (−2)       | 18  |
| 1         | −2         | **−4**               | 18 + (−4)       | 14  |
| 2         | −4         | **−6**               | 14 + (−6)       | 8   |
| 3         | −6         | **−8**               | 8 + (−8)        | 0   |

The values applied to `y` are: −2, −4, −6, −8. These are `g`, `2g`, `3g`, `4g`.

After `t` frames, the value applied in frame `k` is `k × gravity`. The total change in `y` is:

$$\Delta y = g \cdot 1 + g \cdot 2 + g \cdot 3 + \cdots + g \cdot t = g \sum_{k=1}^{t} k$$

That sum is the **triangular number** — every programmer has seen it:

$$\sum_{k=1}^{t} k = \frac{t(t+1)}{2}$$

In Python: `sum(range(t + 1)) == t * (t + 1) // 2`.

So:

$$\Delta y = g \cdot \frac{t(t+1)}{2} = \frac{g}{2}(t^2 + t)$$

For large `t`, the `t` term becomes negligible compared to `t²`:

$$\Delta y \approx \frac{1}{2}g t^2$$

Including an initial vertical velocity `v0` (which contributes `v0` per frame, so `v0 × t` total):

$$\boxed{y(t) = y_0 + v_0 t + \frac{1}{2}g t^2}$$

**The $\frac{1}{2}$ is not a magic constant.** It is the triangular number formula. It emerges from the accumulation in the loop. You have been computing it every time you wrote `dy += gravity`.

In [ ]:
# Verify: discrete loop vs closed-form derivation vs continuous formula

g, y0, v0 = -2, 20, 0

def loop_sim(y0, v0, g, frames):
    """The game loop, exactly."""
    dy, y = float(v0), float(y0)
    positions = []
    for _ in range(frames):
        dy += g
        y  += dy
        positions.append(y)
    return positions

def discrete_formula(y0, v0, g, t):
    """Exact closed form for the discrete loop: y0 + v0*t + g*t*(t+1)/2"""
    return y0 + v0 * t + g * t * (t + 1) / 2

def continuous_formula(y0, v0, g, t):
    """The classic physics formula: y0 + v0*t + (1/2)*g*t^2"""
    return y0 + v0 * t + 0.5 * g * t**2

sim_positions = loop_sim(y0, v0, g, 6)

print(f'{"t":<6}{"loop":<12}{"discrete formula":<22}{"continuous formula":<22}{"Euler error"}')
print('-' * 70)
for t in range(1, 7):
    sim   = sim_positions[t - 1]
    disc  = discrete_formula(y0, v0, g, t)
    cont  = continuous_formula(y0, v0, g, t)
    error = abs(sim - cont)
    print(f'{t:<6}{sim:<12.1f}{disc:<22.1f}{cont:<22.1f}{error:.1f}')

print()
print('Loop == discrete formula: always exact.')
print('Discrete ≈ continuous:    close for small t, converges as step size → 0.')

---

## 4. The Euler Insight

What we wrote in the game loop is a real numerical method. It has a name: **Forward Euler integration**.

The continuous equation for a falling object under constant acceleration is:

$$\frac{d^2y}{dt^2} = g$$

That says: the second derivative of position with respect to time equals gravitational acceleration. A physicist solves this analytically and gets $y = y_0 + v_0 t + \frac{1}{2}gt^2$.

A programmer cannot run a differential equation. They discretise it — break continuous time into small steps of size `dt` and ask: *if I know the state now, what is the state one step later?* Forward Euler answers that question:

```python
y  += dy * dt   # position steps forward by velocity × time-step
dy += g  * dt   # velocity steps forward by acceleration × time-step
```

In our game loop `dt = 1` is implicit — one frame is one time unit, so we wrote `dy += gravity` instead of `dy += gravity * 1`. That is a large step — which is why the discrete formula has that extra `t` term compared to the continuous one. Shrink `dt` and the two converge:

> **Halve the step size → halve the error.** This is what *first-order accurate* means.

The Amiga programmer in 1988 writing `dy += gravity` and the physicist solving a second-order ODE were applying the same approximation. The only difference was step size.

Forward Euler is the simplest integrator and also the least stable — modern physics engines use Verlet or Runge-Kutta for better accuracy. But Euler is the foundation they all build on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

y0, v0, g = 10.0, 0.0, -2.0
T = 3.0

def euler_forward(y0, v0, g, T, dt):
    """Forward Euler: position uses old velocity, then velocity updates."""
    ts, ys = [0.0], [y0]
    y, v, t = y0, v0, 0.0
    while t < T - dt / 2:
        y += v * dt   # old velocity
        v += g * dt   # then update velocity
        t += dt
        ts.append(t)
        ys.append(y)
    return np.array(ts), np.array(ys)

t_exact = np.linspace(0, T, 500)
y_exact = y0 + v0 * t_exact + 0.5 * g * t_exact**2

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.tick_params(colors='#cdd6f4')
for spine in ax.spines.values():
    spine.set_edgecolor('#45475a')

ax.plot(t_exact, y_exact, color='#cdd6f4', linewidth=2.5,
        linestyle='--', label='Exact formula  y = y₀ + v₀t + ½gt²', zorder=5)

steps = [
    (1.0,  '#f38ba8', 'Euler  dt = 1.0   (game-loop frame rate)'),
    (0.5,  '#fab387', 'Euler  dt = 0.5   (2× frame rate)'),
    (0.1,  '#a6e3a1', 'Euler  dt = 0.1   (10× frame rate)'),
]
for dt, color, label in steps:
    ts, ys = euler_forward(y0, v0, g, T, dt)
    ax.plot(ts, ys, 'o-', color=color, linewidth=1.5, markersize=4, label=label)

ax.set_xlabel('t  (time)', color='#cdd6f4')
ax.set_ylabel('y  (position)', color='#cdd6f4')
ax.set_title('Euler convergence — smaller step size → closer to exact formula', color='#cdd6f4', pad=12)
ax.legend(facecolor='#313244', labelcolor='#cdd6f4', framealpha=0.8, fontsize=9)
plt.tight_layout()
plt.show()

# Measure the error at t = T
y_at_T = y0 + v0 * T + 0.5 * g * T**2
print(f'Exact y at t={T}: {y_at_T:.4f}')
print()
for dt, _, label in steps:
    ts, ys = euler_forward(y0, v0, g, T, dt)
    err = abs(ys[-1] - y_at_T)
    print(f'  dt={dt}:  y={ys[-1]:.4f}   error={err:.4f}')

print()
print('Error halves as dt halves — this is first-order (O(dt)) convergence.')

---

## 5. Plot It

Two curves: the straight horizontal path from notebook 01 (constant velocity), and the arc from this notebook (gravity). Plotting position vs time makes the degree difference visible as shape.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 10, 400)

# Notebook 01: x(t) = x0 + v*t  — straight line, constant velocity
x0_nb1, v_nb1 = 0, 3
y_linear = x0_nb1 + v_nb1 * t

# This notebook: y(t) = y0 + v0*t + (1/2)*g*t^2 — parabola
y0_nb2, v0_nb2, g_nb2 = 30, 0, -0.6
y_parabola = y0_nb2 + v0_nb2 * t + 0.5 * g_nb2 * t**2
y_parabola = np.clip(y_parabola, 0, None)  # floor at 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
fig.patch.set_facecolor('#1e1e2e')

def style_ax(ax, title):
    ax.set_facecolor('#1e1e2e')
    ax.tick_params(colors='#cdd6f4')
    ax.set_xlabel('t  (time)', color='#cdd6f4')
    ax.set_ylabel('position', color='#cdd6f4')
    ax.set_title(title, color='#cdd6f4', pad=12)
    for spine in ax.spines.values():
        spine.set_edgecolor('#45475a')

# Left: constant velocity
style_ax(axes[0], 'Notebook 01 — Constant velocity\ndegree 1 · straight line')
axes[0].plot(t, y_linear, color='#89dceb', linewidth=2.5)
axes[0].annotate('x = x₀ + vt', xy=(5, x0_nb1 + v_nb1 * 5),
                 xytext=(6.5, 8), color='#89dceb', fontsize=11,
                 arrowprops=dict(arrowstyle='->', color='#89dceb', lw=1.2))

# Right: parabola
style_ax(axes[1], 'This notebook — With gravity\ndegree 2 · parabola')
axes[1].axhline(0, color='#45475a', linewidth=1.5, linestyle='-')
axes[1].plot(t, y_parabola, color='#a6e3a1', linewidth=2.5)
# Mark the floor-hit point
t_floor = np.sqrt(2 * y0_nb2 / abs(g_nb2))
axes[1].axvline(t_floor, color='#45475a', linewidth=1, linestyle=':')
axes[1].annotate(f'lands at t≈{t_floor:.1f}', xy=(t_floor, 0.5),
                 xytext=(t_floor + 0.4, 8), color='#6c7086', fontsize=9,
                 arrowprops=dict(arrowstyle='->', color='#6c7086', lw=1.0))
axes[1].annotate('y = y₀ + v₀t + ½gt²', xy=(3, y0_nb2 + v0_nb2*3 + 0.5*g_nb2*9),
                 xytext=(4, 20), color='#a6e3a1', fontsize=11,
                 arrowprops=dict(arrowstyle='->', color='#a6e3a1', lw=1.2))

plt.tight_layout()
plt.show()

---

## 6. Worked Examples

Using the formula $y = y_0 + v_0 t + \frac{1}{2}gt^2$ directly. Each example is evaluated step by step.

**Example 1** — Ball dropped from rest: $y_0 = 30$, $v_0 = 0$, $g = -2$, at $t = 4$:

$$y = 30 + 0 \cdot 4 + \tfrac{1}{2}(-2)(4^2) = 30 + 0 - 16 = 14$$

**Example 2** — Ball thrown upward: $y_0 = 5$, $v_0 = 10$, $g = -2$, at $t = 3$:

$$y = 5 + 10 \cdot 3 + \tfrac{1}{2}(-2)(3^2) = 5 + 30 - 9 = 26$$

**Example 3** — When does it land? $y_0 = 20$, $v_0 = 0$, $g = -2$, find $t$ when $y = 0$:

$$0 = 20 + 0 \cdot t + \tfrac{1}{2}(-2)t^2 = 20 - t^2$$

$$t^2 = 20 \qquad t = \sqrt{20} \approx 4.47$$

This is the kind of question you can answer with the formula before writing any simulation — or use to verify that your simulation is correct.

In [ ]:
import math

def y_at(y0, v0, g, t):
    return y0 + v0 * t + 0.5 * g * t**2

def simulate(y0, v0, g, target_t):
    """Game-loop simulation: run until target_t frames."""
    dy, y = float(v0), float(y0)
    for _ in range(int(target_t)):
        dy += g
        y  += dy
    return y

examples = [
    ('Ex 1  dropped from rest',  30, 0,  -2, 4),
    ('Ex 2  thrown upward',       5, 10, -2, 3),
    ('Ex 3  check landing time',  20, 0,  -2, int(math.sqrt(20))),
]

print(f'{"Example":<28} {"formula":>10}  {"simulation":>12}  {"match"}')
print('-' * 60)
for label, y0, v0, g, t in examples:
    formula_result = y_at(y0, v0, g, t)
    sim_result     = simulate(y0, v0, g, t)
    close = abs(formula_result - sim_result) < 2   # allow small Euler error
    print(f'{label:<28} {formula_result:>10.2f}  {sim_result:>12.2f}  {"✓" if close else "check"}')

print()
print('Ex 3 — exact landing time from formula:')
t_land = math.sqrt(2 * 20 / 2)   # solve 20 - t^2 = 0
print(f'  t = sqrt(2 × y0 / |g|) = sqrt({2*20}/{2}) = sqrt({2*20//2}) ≈ {t_land:.4f}')

---

## Exercises

Go back to the animation cell and experiment:

- **Set `gravity = 0`** — what happens? What does the formula $y = y_0 + v_0 t + \frac{1}{2}(0)t^2$ reduce to? Is this notebook 01 again?
- **Set `gravity` to a positive number** — which direction does the ball now accelerate? What does a positive `gravity` represent physically?
- **Give the ball an initial upward velocity** — change the initial `dy` to a positive value before the animation starts. At what point does the ball stop rising? What does the formula predict for the peak?
- **Predict the landing frame** — using Example 3 as a template, calculate from the formula when your ball should hit the floor. Then count the animation frames to verify.
- **Increase the frame rate (smaller step)** — modify the Euler convergence cell. At what step size does the simulation match the formula to within 0.01?

---

## Summary

| Concept | Plain English | Code | Math |
|---------|--------------|------|------|
| Acceleration | velocity changes each frame | `dy += gravity` | $\Delta v = g \cdot \Delta t$ |
| Accumulated velocity | `dy` is a running total, not a constant | `dy` grows every frame | $v(t) = v_0 + gt$ |
| Quadratic position | position changes at a changing rate | `y += dy` where `dy` itself changes | $y = y_0 + v_0 t + \frac{1}{2}gt^2$ |
| The ½ | comes from summing consecutive integers | `sum(range(t+1)) == t*(t+1)//2` | $\sum_{k=1}^{t} k = \frac{t(t+1)}{2} \approx \frac{t^2}{2}$ |
| Euler integration | discrete approximation of a continuous formula | the game loop | Forward Euler method |
| Degree 1 vs 2 | straight line vs curve | one accumulating variable | $t$ vs $t^2$ |

**The key insight:** the $\frac{1}{2}gt^2$ term does not appear anywhere in the code. It emerges from the accumulation. Every time you write `dy += gravity`, you are computing a discrete integral — and the triangular number formula is what makes it quadratic.

---

**Next:** [03 — Bounce](./03-bounce.ipynb) — what happens when the ball hits the floor and the velocity reverses?